# 02. TF-IDF Baseline

Enhanced TF-IDF with handcrafted features + calibration.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

# Target
train_df['target'] = (train_df['winner_model_a'].astype(int) * 0 +
                      train_df['winner_model_b'].astype(int) * 1 +
                      train_df['winner_tie'].astype(int) * 2)

In [ ]:
# TF-IDF features
train_text = (train_df['prompt'].fillna('') + ' ' +
               train_df['response_a'].fillna('') + ' ' +
               train_df['response_b'].fillna(''))
test_text = (test_df['prompt'].fillna('') + ' ' +
              test_df['response_a'].fillna('') + ' ' +
              test_df['response_b'].fillna(''))

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(train_text)
X_test_tfidf = vectorizer.transform(test_text)

print(f'TF-IDF shape: {X_train_tfidf.shape}')

In [ ]:
# Handcrafted features
def handcrafted_features(df):
    feats = pd.DataFrame()
    feats['len_a'] = df['response_a'].str.len()
    feats['len_b'] = df['response_b'].str.len()
    feats['len_ratio'] = feats['len_a'] / (feats['len_b'] + 1)
    feats['word_count_a'] = df['response_a'].str.split().str.len()
    feats['word_count_b'] = df['response_b'].str.split().str.len()
    feats['word_ratio'] = feats['word_count_a'] / (feats['word_count_b'] + 1)
    feats['prompt_len'] = df['prompt'].str.len()
    feats['total_len'] = feats['len_a'] + feats['len_b']
    return feats.fillna(0).values

X_train_hand = handcrafted_features(train_df)
X_test_hand = handcrafted_features(test_df)

# Combine
X_train = hstack([X_train_tfidf, X_train_hand])
X_test = hstack([X_test_tfidf, X_test_hand])
y_train = train_df['target'].values

print(f'Combined shape: {X_train.shape}')

In [ ]:
# CV evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X_train), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', multi_class='multinomial')
    model.fit(X_train[train_idx], y_train[train_idx])
    oof_preds[val_idx] = model.predict_proba(X_train[val_idx])
    fold_loss = log_loss(y_train[val_idx], oof_preds[val_idx])
    print(f'Fold {fold+1}: log_loss={fold_loss:.4f}')

oof_loss = log_loss(y_train, oof_preds)
print(f'\nOOF log_loss: {oof_loss:.4f}')

In [ ]:
# Final model + predictions
model = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', multi_class='multinomial')
model.fit(X_train, y_train)
test_preds = model.predict_proba(X_test)

print(f'Test predictions shape: {test_preds.shape}')
print(f'Row sums: {test_preds.sum(axis=1)[:5]}')

In [ ]:
# Save
np.save(OUTPUT_DIR / 'tfidf_oof.npy', oof_preds)
np.save(OUTPUT_DIR / 'tfidf_test.npy', test_preds)
print('Saved TF-IDF predictions')